In [5]:
import pandas as pd
import numpy as np
from scipy import stats

orders = pd.read_csv('data/orders.csv')
customers = pd.read_csv('data/customers.csv')

CUTOFF, HORIZON = 25, 6
active_ids = orders[(orders['week'] > CUTOFF - 6) &
                    (orders['week'] <= CUTOFF)]['customer_id'].unique()
exp = customers[customers['in_experiment'] & customers['customer_id'].isin(active_ids)].copy()
print('Customers in the experiment:', len(exp))

Customers in the experiment: 321


In [6]:
future_ids = set(orders[(orders['week'] > CUTOFF) &
                        (orders['week'] <= CUTOFF + HORIZON)]['customer_id'].unique())
exp['reordered'] = exp['customer_id'].isin(future_ids).astype(int)

summary = exp.groupby('got_discount')['reordered'].agg(['mean', 'count'])
summary.columns = ['reorder_rate', 'customers']
summary

,reorder_rate,customers
got_discount,,
False,0.553333,150
True,0.760234,171


In [7]:

treat = exp[exp['got_discount']]
ctrl  = exp[~exp['got_discount']]

x1, n1 = treat['reordered'].sum(), len(treat)   # discount group
x2, n2 = ctrl['reordered'].sum(),  len(ctrl)    # control group
p1, p2 = x1 / n1, x2 / n2


p_pool = (x1 + x2) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))   # standard error
z  = (p1 - p2) / se                                   # how many SEs apart
p_value = 2 * (1 - stats.norm.cdf(abs(z)))            # two-sided p-value

print(f'Discount reorder rate : {p1:.3f}')
print(f'Control  reorder rate : {p2:.3f}')
print(f'Difference            : {p1 - p2:+.3f}')
print(f'z-score               : {z:.2f}')
print(f'p-value               : {p_value:.4f}')

Discount reorder rate : 0.760
Control  reorder rate : 0.553
Difference            : +0.207
z-score               : 3.91
p-value               : 0.0001


In [8]:
if p_value < 0.05:
    print(f'The discount lifted reorders by {(p1-p2)*100:.1f} percentage points.')
    print('This is statistically significant (p < 0.05): unlikely to be chance.')
    print('Recommendation: roll the discount out more widely and keep measuring.')
else:
    print('We cannot conclude the discount helped (p >= 0.05).')
    print('Recommendation: keep testing with a larger group before deciding.')

The discount lifted reorders by 20.7 percentage points.
This is statistically significant (p < 0.05): unlikely to be chance.
Recommendation: roll the discount out more widely and keep measuring.
